# A-Share Stock Price Analysis
This project retrieves historical price data for a selected A-share stock, calculates key metrics including moving averages and rolling volatility, and visualises price trends.

In [ ]:
import akshare as ak
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── 1. Retrieve historical data ──────────────────────────────────────────────
# Using Ping An Bank (000001) as an example; change the symbol as needed
symbol = "000001"
df = ak.stock_zh_a_hist(symbol=symbol, period="daily",
                        start_date="20220101", end_date="20241231",
                        adjust="qfq")  # qfq = forward-adjusted prices

# Rename columns for convenience
df = df.rename(columns={
    "日期": "date",
    "开盘": "open",
    "收盘": "close",
    "最高": "high",
    "最低": "low",
    "成交量": "volume"
})
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()

print(f"Data loaded: {len(df)} trading days")
df.head()

In [ ]:
# ── 2. Calculate metrics ─────────────────────────────────────────────────────
# Moving averages
df["MA20"]  = df["close"].rolling(window=20).mean()
df["MA60"]  = df["close"].rolling(window=60).mean()

# Daily returns
df["daily_return"] = df["close"].pct_change()

# 20-day rolling volatility (annualised)
df["volatility"] = df["daily_return"].rolling(window=20).std() * (252 ** 0.5)

print("Metrics calculated.")
df[["close", "MA20", "MA60", "volatility"]].tail(10)

In [ ]:
# ── 3. Visualise ─────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                gridspec_kw={"height_ratios": [3, 1]},
                                sharex=True)
fig.suptitle(f"Stock {symbol} — Price & Volatility Analysis", fontsize=14)

# Price + moving averages
ax1.plot(df.index, df["close"],  color="steelblue", linewidth=1,   label="Close price")
ax1.plot(df.index, df["MA20"],   color="orange",    linewidth=1.2, label="MA20")
ax1.plot(df.index, df["MA60"],   color="green",     linewidth=1.2, label="MA60")
ax1.set_ylabel("Price (CNY)")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.3)

# Rolling volatility
ax2.fill_between(df.index, df["volatility"], color="salmon", alpha=0.5, label="20-day volatility (ann.)")
ax2.set_ylabel("Volatility")
ax2.set_xlabel("Date")
ax2.legend(loc="upper left")
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig("stock_analysis.png", dpi=150)
plt.show()
print("Chart saved as stock_analysis.png")

In [ ]:
# ── 4. Summary statistics ────────────────────────────────────────────────────
summary = {
    "Period":              f"{df.index[0].date()} to {df.index[-1].date()}",
    "Trading days":        len(df),
    "Starting price":      f"{df['close'].iloc[0]:.2f}",
    "Ending price":        f"{df['close'].iloc[-1]:.2f}",
    "Total return":        f"{(df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100:.1f}%",
    "Average daily return":f"{df['daily_return'].mean() * 100:.3f}%",
    "Avg annualised vol":  f"{df['volatility'].mean() * 100:.1f}%",
    "Max drawdown date":   str(df['close'].idxmin().date()),
}

print("── Summary ──────────────────────────────")
for k, v in summary.items():
    print(f"  {k:<25} {v}")